# Modular division

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.modular_inverse_catalog import modular_inverse_examples
from examples.algorithms.register_modular_inverse import ModularInverse
from flagquantum.compiler.registers.modular.modular_inverse import (
    modular_inverse_outputs, source_modular_inverse_t_count,
)
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## `KaliskiModInverse`
Compute the in-place modular inverse of a Montgomery-encoded integer.

Applies the transformation
$$
    \ket{x} \ket{0} \rightarrow \ket{x^{-1} 2^{2n} \mod p} \ket{\mathrm{garbage}}
$$

#### Parameters
 - `bitsize`: size of the number.
 - `mod`: The integer modulus.
 - `uncompute`: whether to compute or uncompute. 

#### Registers
 - `x`: The register for which we compute the multiplicative inverse.
 - `junk` is the 4n-bit history. Its upper 2n bits record the branch markers and its lower 2n bits record the terminal markers.

#### References
 - [Performance Analysis of a Repetition Cat Code Architecture: Computing 256-bit Elliptic Curve Logarithm in 9 Hours with 126 133 Cat Qubits](https://arxiv.org/abs/2302.06639).     Appendix C5.
 - [Improved quantum circuits for elliptic curve discrete logarithms](https://arxiv.org/abs/2001.09580).     Fig 7(b)
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585).     page 8.


In [ ]:
examples = modular_inverse_examples()

### Example Instances

In [ ]:
concrete = examples["kaliskimodinverse_example"]

In [ ]:
symbolic = examples["kaliskimodinverse_symbolic"]

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("concrete", concrete.signature(), {}),
    ("symbolic", symbolic.signature(), {}),
    ("adjoint", concrete.signature(adjoint=True), {}),
])))

### Call Graph

In [ ]:
graph = concrete.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)
display(source_modular_inverse_t_count(symbolic.width))

### Native execution

The four-width history word stays live across compute and uncompute. The forward result and clean work exchange physical roles, so the inverse consumes the returned layout.
This example constructs the measured primitive circuits and reports their actual gate counts.

In [ ]:
n, modulus = 3, 5
width = 10 * n + 8
circuit = DynamicCircuit(width).append(ModularInverse(n, modulus))
display(Counter(gate.name for gate in circuit.to_ir().instructions))

target, work = tuple(range(n)), tuple(range(5 * n, width))
inverse_input, returned_work = modular_inverse_outputs(target, work)
display({"result_wires": inverse_input, "history_bits": 4 * n})
assert modular_inverse_outputs(inverse_input, returned_work, adjoint=True) == (target, work)
circuit.append(ModularInverse(n, modulus, adjoint=True))